1️⃣ Import Libraries

In [ ]:
# pandas — the main library for working with data tables (DataFrames)
import pandas as pd
# numpy — for numerical operations and handling infinity/NaN values
import numpy as np
# matplotlib — for plotting charts and graphs
import matplotlib.pyplot as plt
# seaborn — for statistical data visualization
import seaborn as sns
# warnings — hide unnecessary output so our notebook stays clean
import warnings
warnings.filterwarnings('ignore') # 'ignore' means don't show warnings

2️⃣ Load & Explore the Dataset

In [ ]:
# pd.read_csv() reads the CSV file and converts it into a DataFrame
# A DataFrame is like an Excel table inside Python
df = pd.read_csv('/content/Case_Data.csv')
# .shape tells us (number of rows, number of columns)
print('Shape:', df.shape)
# .head() shows the first 5 rows so we can see what the data looks like
df.head()

In [ ]:
# .isnull() checks every cell — True if missing, False if not
# .sum() counts how many True values (missing) are in each column
df.isnull().sum()

3️⃣ Data Cleaning

### 3.1 Drop Columns with Excessive Missing Values

First, let's identify and drop columns that have a very high percentage of missing values, as they might not be useful for analysis or modeling.

In [ ]:
missing_percentages = df.isnull().sum() / len(df) * 100
columns_to_drop = missing_percentages[missing_percentages > 50].index.tolist()

print(f"Columns to drop due to high missing values (>50%): {columns_to_drop}")
df = df.drop(columns=columns_to_drop)

print(f"Shape after dropping columns: {df.shape}")

### 3.2 Convert Data Types and Clean Specific Columns

Next, we'll convert columns to their appropriate data types and clean specific columns that require reformatting (e.g., removing '%' from percentage columns).

In [ ]:
# Convert 'term' to numerical (months) if it's an object/string type
if df['term'].dtype == 'object':
    df['term'] = df['term'].str.replace(' months', '').astype(int)

# 'int_rate' and 'revol_util' are already numeric (float64), so no .str.replace('%', '') is needed.
# If they were strings with '%', the code would be:
# if df['int_rate'].dtype == 'object':
#     df['int_rate'] = df['int_rate'].str.replace('%', '', regex=False).astype(float)
# if 'revol_util' in df.columns and df['revol_util'].dtype == 'object':
#     df['revol_util'] = df['revol_util'].str.replace('%', '', regex=False).astype(float)

# Convert date columns to datetime objects
date_cols = ['issue_d', 'earliest_cr_line', 'last_pymnt_d', 'last_credit_pull_d']
for col in date_cols:
    if col in df.columns:
        # Only convert if the column is currently an object (string) and not already datetime
        if df[col].dtype == 'object':
            df[col] = pd.to_datetime(df[col], format='%b-%Y', errors='coerce')

print("Data types converted for 'term' and date columns (if needed). 'int_rate' and 'revol_util' were already numeric.")

### 3.3 Handle Remaining Missing Values

We'll fill remaining missing numerical values with the median and categorical values with the mode to ensure a complete dataset.

In [ ]:
# Impute numerical columns with median
for col in df.select_dtypes(include=np.number).columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# Impute categorical columns with mode or 'Unknown'
for col in df.select_dtypes(include='object').columns:
    if df[col].isnull().any():
        # For 'emp_title', 'emp_length', 'title', 'purpose', 'loan_status' fill with 'Unknown'
        if col in ['emp_title', 'emp_length', 'title', 'purpose', 'loan_status']:
            df[col] = df[col].fillna('Unknown')
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

print("Remaining missing values imputed.")

### 3.4 Final Check After Cleaning

Let's check the missing values again and display the head of the DataFrame to see the cleaned data.

In [ ]:
print('Missing values after cleaning:')
print(df.isnull().sum().sum())

print('\nDataFrame Info after cleaning:')
df.info()

print('\nFirst 5 rows of the cleaned DataFrame:')
display(df.head())

5️⃣ Exploratory Data Analysis (EDA)

### 5.1 Distribution of Loan Amount and Interest Rate

Let's start by visualizing the distribution of the key financial variables: `loan_amnt` and `int_rate`.

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
sns.histplot(df['loan_amnt'], bins=30, kde=True)
plt.title('Distribution of Loan Amount')
plt.xlabel('Loan Amount')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
sns.histplot(df['int_rate'], bins=30, kde=True)
plt.title('Distribution of Interest Rate')
plt.xlabel('Interest Rate (%)')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

### 5.2 Distribution of Loan Status

Understanding the distribution of `loan_status` is crucial as it's often the target variable in loan prediction models.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(y=df['loan_status'], order=df['loan_status'].value_counts().index)
plt.title('Distribution of Loan Status')
plt.xlabel('Count')
plt.ylabel('Loan Status')
plt.show()

### 5.3 Distribution of Employment Length and Home Ownership

These categorical features can provide insights into the borrower's stability.

In [ ]:
plt.figure(figsize=(18, 6))

plt.subplot(1, 2, 1)
sns.countplot(y=df['emp_length'], order=df['emp_length'].value_counts().index)
plt.title('Distribution of Employment Length')
plt.xlabel('Count')
plt.ylabel('Employment Length')

plt.subplot(1, 2, 2)
sns.countplot(y=df['home_ownership'], order=df['home_ownership'].value_counts().index)
plt.title('Distribution of Home Ownership')
plt.xlabel('Count')
plt.ylabel('Home Ownership')

plt.tight_layout()
plt.show()

6️⃣ Save Cleaned Dataset

In [ ]:
# Save the cleaned DataFrame to a new CSV file
df.to_csv('cleaned_case_data.csv', index=False)
print("Cleaned dataset saved to 'cleaned_case_data.csv'")

LOAN RISK ANALYSIS PREDICTOR

1️⃣ Import Libraries

In [ ]:
import pickle
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

2️⃣ Load the Cleaned Dataset

In [ ]:
df = pd.read_csv('/content/cleaned_case_data.csv')
print('Shape:', df.shape)
df.head()

3️⃣ Train / Test Split

In [ ]:
X = df.drop('loan_status', axis=1)
y = df['loan_status']
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.25, random_state=42)
print('Training rows:', len(X_train))
print('Testing rows: ', len(X_test))

4️⃣ Preprocessing and Model Training

### 4.1 Identify Feature Types

First, we need to separate our features into numerical and categorical types so we can apply appropriate preprocessing steps to each.

In [ ]:
# Columns identified as being entirely NaT (Not a Time) from previous df.info() output
date_cols_to_drop = ['issue_d', 'earliest_cr_line', 'last_pymnt_d', 'last_credit_pull_d']

# Drop these columns from X_train and X_test
X_train = X_train.drop(columns=date_cols_to_drop, errors='ignore')
X_test = X_test.drop(columns=date_cols_to_drop, errors='ignore')

# Now identify numerical and categorical columns from the updated X_train
numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()

print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

### 4.2 Create a Preprocessing Pipeline

We'll use `ColumnTransformer` to apply `StandardScaler` to numerical features and `OneHotEncoder` to categorical features.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

print("Preprocessing pipeline created.")

### 4.3 Build and Train the Model Pipeline

Now we'll combine the preprocessor with a `LogisticRegression` model into a single pipeline and train it on our training data.

In [ ]:
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', LogisticRegression(random_state=42, solver='liblinear'))])

model_pipeline.fit(X_train, y_train)

print("Model training complete.")

5️⃣ Evaluate the Model

In [ ]:
y_pred = model_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy on Test Set: {accuracy:.4f}")

6️⃣ Save the Trained Model

In [ ]:
with open('loan_risk_model.pkl', 'wb') as file:
    pickle.dump(model_pipeline, file)
print("Trained model saved as 'loan_risk_model.pkl'")